# Spotify sob a Lupa: Audiência Gratuita ou Lucro Premium? 
**Daniel Amorim** | Marketing Technologist  
[Website](https://damorimdigital.com) • [LinkedIn](https://linkedin.com/in/danieloamorim/) • [Email](mailto:daniel@damorimdigital.com)

---

### O Ponto de Partida
Sempre tive curiosidade em entender como o Spotify equilibra o serviço gratuito com o modelo de assinaturas.  
Para investigar isso, decidi abrir o relatório financeiro oficial 20-F de 2025 e analisar os números reais através de automação em Python.
  
O meu objetivo não é apenas comparar dois planos, mas avaliar a saúde do ecossistema da empresa.  
Procuro entender como o plano gratuito funciona como um "Gerador de Audiência",  
onde o utilizador não paga com dinheiro, mas sim com a atenção, que é vendida a anunciantes.
      
**Premissa da análise:** No plano gratuito, o usuário não é exatamente o cliente, 
ele é o **produto**.  
O verdadeiro cliente desse segmento são as marcas que compram essa audiência.  
    
Quero descobrir se vender "Audiência" é um negócio viável 
ou se a empresa depende exclusivamente das assinaturas.

In [ ]:
import pandas as pd
import plotly.express as px

# Configurações padrão para todos os gráficos:
layout_padrao = dict(
    width=600,
    height=400,
    title_x=0.5,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=50, r=50, t=80, b=100),
    font=dict(family="Arial", size=12)
)
# Definir as cores dos gráficos:
cor_spotify = '#1DB954'
cor_cinza = "#313131"

# Caminho do arquivo extraído do formulário SEC 20-F:
file_path = "spotify_20f_2025.xls"

print("Ambiente configurado. Dados prontos para análise.")

Ambiente configurado. Dados prontos para análise.


### Escolha de Métricas

Para que a análise seja justa, foquei em três pilares fundamentais:  
* **Receita:** O montante total que entra na empresa.  
* **Custo:** O que o Spotify gasta para manter o serviço a funcionar (licenças, servidores, suporte).  
* **Volume de Utilizadores (MAUs):** "estoque de audiência", fundamental para calcular a eficiência real da operação.
  
É através do cruzamento entre estes dados que identificamos quanto valor cada utilizador gera.  
Para isso, calculei três indicadores complementares:

* **Lucro Bruto:** A diferença entre a receita e o custo direto.  
* **Margem Bruta (%):** Que percentagem da receita sobra após pagar os custos essenciais.
* **ARPU (Average Revenue Per User):** A receita média mensal por utilizador.  

> **Nota importante sobre o ARPU:** Embora preços como os **8,99€** do plano individual em Portugal nos venham logo à cabeça, o valor no relatório 20-F é uma média global.  
Ele parece mais baixo porque reflete a realidade de todo o mundo: planos familiares divididos, descontos para estudantes e preços adaptados a países com moedas mais fracas.  
É o "valor real" que entra no caixa do Spotify por cada cabeça, na média de todos os mercados.
  
**Metodologia de Extração:** Os dados operacionais e financeiros foram extraídos das Tabelas 4, 5, 6 e 7 do relatório oficial.  
Optei por utilizar as tabelas segmentadas por plano (Premium vs Ad-Supported) para garantir que as métricas de conversão fossem o mais precisas possível,  
reduzindo erros de transcrição manual através da automação com Python.

In [2]:
# Leitura das tabelas de Receita e Custo:
tab_receita = pd.read_excel(file_path, sheet_name="TABLE6", header=None)
tab_custo = pd.read_excel(file_path, sheet_name="TABLE7", header=None)

# Leitura das tabelas de Usuários:
tab_premium_maus = pd.read_excel(file_path, sheet_name="TABLE4", header=None)
tab_ads_maus = pd.read_excel(file_path, sheet_name="TABLE5", header=None)

# Extração dos valores exatos (Coluna 2, conforme investigação):
mau_premium = float(tab_premium_maus.iloc[12, 2]) # Linha 12 da TABLE4
mau_ads = float(tab_ads_maus.iloc[24, 2])         # Linha 24 da TABLE5

# Consolidação do DataFrame:
df = pd.DataFrame({
    'Segmento': ['Premium', 'Ad-Supported'],
    'Receita': [float(tab_receita.iloc[16, 2]), float(tab_receita.iloc[17, 2])],
    'Custo_Direto': [float(tab_custo.iloc[14, 2]), float(tab_custo.iloc[15, 2])],
    'MAUs': [mau_premium, mau_ads] 
})

# Cálculo de Performance:
df['Lucro_Bruto'] = df['Receita'] - df['Custo_Direto']
df['Margem_%'] = (df['Lucro_Bruto'] / df['Receita']) * 100
df['ARPU_Mensal'] = (df['Receita'] / 12) / df['MAUs']

# Exibição:
df_clean = df.style.format({
    'Receita': '€{:,}M', 
    'Custo_Direto': '€{:,}M', 
    'MAUs': '{:,}M',
    'Lucro_Bruto': '€{:,}M', 
    'Margem_%': '{:.2f}%',
    'ARPU_Mensal': '€{:.2f}'
}).hide(axis='index')

display(df_clean)

Segmento,Receita,Custo_Direto,MAUs,Lucro_Bruto,Margem_%,ARPU_Mensal
Premium,"€15,350.0M","€10,184.0M",290.0M,"€5,166.0M",33.65%,€4.41
Ad-Supported,"€1,836.0M","€1,506.0M",476.0M,€330.0M,17.97%,€0.32


**Comentário:**  
Ao olhar para essa tabela, consegui entender como o dinheiro funciona dentro do Spotify.  
  
- O lucro de cada lado: Notei que o plano Premium "guarda" muito mais dinheiro. De cada 100 euros que entram no Premium, sobram quase 34 euros. Já no plano gratuito, só sobram 18 euros.
- O valor de cada pessoa: Fiz um cálculo para saber quanto cada usuário rende por mês. Descobri que um assinante rende €4,41, enquanto quem ouve anúncios rende só €0,32.
- A diferença é significativa: Na prática, O Spotify precisa de 14 pessoas ouvindo anúncios para ganhar o mesmo que ganha com apenas 1 pessoa pagando

Isso mostra que, por mais que o plano gratuito tenha muita gente, o Spotify precisa que esses usuários virem assinantes para o negócio valer a pena.

### Análise da Venda de Audiência
Para identificar se o plano gratuito é lucrativo, é preciso 
observar a **Margem %**. 

Notei que, embora o Spotify atraia 476 milhões de pessoas para verem anúncios, a sobra financeira é de apenas **18%**.

Isso ocorre porque os custos diretos da operação (que incluem licenciamento, servidores e infraestrutura)  
consomem a maior parte da receita gerada por publicidade.

Já no Premium, a margem é de **33.7%**. Isso mostra que 
vender assinaturas é quase duas vezes mais eficiente 
do que vender a atenção do usuário para terceiros.

In [3]:
# Preparação dos dados:
df_plot = df.melt(id_vars='Segmento', value_vars=['Receita', 'Custo_Direto'], 
                  var_name='Métrica', value_name='Valor')

fig1 = px.bar(df_plot, x='Segmento', y='Valor', color='Métrica', barmode='group',
             color_discrete_sequence=[cor_spotify, cor_cinza],
             title='Equilíbrio Financeiro: Onde a operação é mais eficiente?')

# Padrão e ajustes gráficos:
fig1.update_layout(**layout_padrao)
fig1.update_layout(
    yaxis_title='Euros (Milhões)', 
    xaxis_title=None,
    legend=dict(orientation="h", yanchor="bottom", y=-0.3, xanchor="center", x=0.5)
)
fig1.show()

**Comentário:**  
Depois de calcular os valores, decidi visualizar a diferença entre a Receita (o que entra) e o Custo (o que sai) para cada lado do negócio.  

- A distância entre as barras:  
No lado Premium, a barra verde (dinheiro que entra) é bem mais alta que a cinza (custo).  
Isso significa que sobra uma "fatia" muito boa de lucro para a empresa investir nela mesma.
  
- O sufoco do plano gratuito:  
Já no lado Ad-Supported (quem ouve anúncios), as duas barras estão quase na mesma altura.  
Isto confirma que os custos necessários para manter este serviço a funcionar (como o licenciamento de conteúdos e a infraestrutura técnica)  
consomem grande parte do que é ganho com publicidade.
  
> **Conclusão:** Esse gráfico deixa claro que, embora o plano gratuito seja enorme em número de pessoas, ele mal se paga.  
Quem realmente mantém o Spotify de pé e dá lucro de verdade é o assinante.

### O Papel Estratégico do Funil
Se a venda de anúncios rende pouco, por que manter uma 
base tão grande? Analiso isso como um **Funil de Vendas**. 

O plano gratuito atrai o volume de pessoas, mas o lucro 
real que sustenta a empresa vem do assinante Premium. 

O gráfico a seguir prova que o sucesso do negócio depende 
da velocidade com que a marca realiza o **Upsell** (a conversão do usuário gratuito em pagante).

In [4]:
fig_funnel = px.funnel(df, x='MAUs', y='Segmento', 
                       title='Funil de Conversão: Volume de Utilizadores',
                       category_orders={'Segmento': ['Ad-Supported', 'Premium']},
                       color_discrete_sequence=[cor_spotify])

# Padrão e limpeza de eixos:
fig_funnel.update_layout(**layout_padrao)
fig_funnel.update_traces(textposition='inside', textfont=dict(size=14, color='white'), textinfo='value')
fig_funnel.update_layout(showlegend=False, yaxis_title=None, xaxis_visible=False)
fig_funnel.show()

**Comentário:**  

Este gráfico foca na distribuição de utilizadores e ilustra a estrutura da plataforma.  
É possível observar uma estrutura piramidal: existe uma base massiva de pessoas no plano gratuito (476M) e, a partir dela,  
um volume significativo (290M) consolida-se no plano Premium.
  
**O que o funil indica:**  
O esforço do Marketing é eficaz em atração de volume de novos utilizadores,  
o desafio seguinte passa por otimizar a conversão para o plano pago, onde o valor gerado para a empresa é superior.

### Origem da Sustentabilidade Financeira
Se o plano gratuito concentra a maior parte dos utilizadores, de onde vem o lucro que realmente sustenta a operação?
  
Será analisada a distribuição do lucro bruto para entender o peso real de cada segmento.  
O resultado indica que a saúde financeira da empresa não é proporcional ao volume de pessoas, mas sim à categoria de utilizador.
  
O gráfico a seguir revela que a estabilidade do negócio depende quase exclusivamente do segmento Premium,  
reforçando a importância de converter a audiência em subscritores fiéis.

In [5]:
fig2 = px.pie(df, values='Lucro_Bruto', names='Segmento', 
             title='Distribuição do Lucro Bruto Total (2025)',
             color_discrete_sequence=[cor_spotify, cor_cinza],
             hole=.4)

# Padrão e destaque das fatias:
fig2.update_layout(**layout_padrao)
fig2.update_traces(textinfo='percent+label', pull=[0.1, 0], textfont_size=14)
fig2.update_layout(showlegend=False)
fig2.show()

**Comentário:**  
Para encerrar a análise, utilizei o gráfico de rosca para ver de onde vem o lucro bruto que realmente sobra para o Spotify.  
  
O Premium sustenta tudo: Notei que 94% do lucro da empresa vem dos assinantes.  
Isso me mostra que o modelo de assinaturas é o verdadeiro coração financeiro do negócio.
  
O papel do plano gratuito: A fatia cinza, que representa quem ouve anúncios, contribui com apenas 6% do lucro.  
Isso me confirmou que o plano gratuito não serve para dar lucro direto, mas sim para atrair pessoas e convencê-las a virarem assinantes no futuro.
  
Minha conclusão visual: No fim das contas, o Spotify usa o plano gratuito como uma vitrine gigante para atrair usuários,  
mas o sucesso do negócio depende totalmente de transformar esses ouvintes em clientes pagantes.

---
## Próximos Passos: Do Insight à Estratégia

Esta análise de dados revelou uma realidade:  
O Spotify opera uma "vitrine gigante" através do plano gratuito, mas a sua sustentabilidade financeira reside na conversão para o segmento Premium. 

Foi identificado:
* A base gratuita gera **volume**, mas apenas **18% de margem**.
* O lucro real (94% do total) é sustentado pelos assinantes.
* O sucesso do ecossistema depende da **velocidade do Upsell**.

### O que vem a seguir?
Os dados deram-nos o diagnóstico; agora é tempo de desenhar a solução.  
  
Na Parte 2 deste projeto, utilizarei estes fundamentos para construir um Plano de Marketing Estratégico. Vou explorar variáveis que os números brutos sugerem, mas não revelam totalmente, tais como:
  
**Retenção e Churn:** Como manter o utilizador Premium após a conversão?  
**Eficiência de Aquisição:** O plano gratuito é um gerador de lucro ou um custo de marketing necessário?  
**Estratégias de Upsell:** Como acelerar a jornada do "Ouvinte" para o "Cliente".  
  
O objetivo será responder ao maior desafio da plataforma: Como otimizar a fluidez do funil e acelerar a conversão de audiência em lucro real?
  
---
**Fica atento à próxima publicação para veres a estratégia completa fundamentada nestes dados.**

**Website:** [damorimdigital.com](https://damorimdigital.com)  
**LinkedIn:** [linkedin.com/in/danieloamorim/](https://www.linkedin.com/in/danieloamorim/)  
**Contacto:** daniel@damorimdigital.com

**Daniel Amorim** *Marketing Technologist | Especialista em Brand & Performance Analytics*